# 38. MCP와 Multi-Agent

> **제38장** · **이론편 대응: 25.3~25.4절 (MCP, Multi-Agent)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음 (개념과 구조 구현 중심)
> **API 키**: 선택

---

## 이 장에서 하는 일

37장에서 도구를 직접 만들어 붙였다. 그런데 **도구가 많아지고 여러 앱에서 쓰려면** 문제가 생긴다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **M×N 문제** | 25.3절 |
| 2 | MCP의 구조 | 25.3절 |
| 3 | **JSON-RPC 메시지 직접 만들기** ★ | 25.3절 |
| 4 | 세 가지 프리미티브 | 25.3절 |
| 5 | 간이 MCP 서버 구현 | 25.3절 |
| 6 | **Multi-Agent — 왜 나누는가** | 25.4절 |
| 7 | 협업 패턴 세 가지 | 25.4절 |
| 8 | 비용과 한계 | 25.4절 |

**3절과 5절에서 직접 만든다.** 라이브러리를 쓰기 전에
"메시지가 어떻게 오가는가"를 손으로 확인한다.

In [ ]:
import json
import inspect
import uuid
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

print("준비 완료")
print()
print("이 장은 프로토콜 구조를 직접 구현해 이해하는 것이 목적이다.")
print("실제 MCP 서버를 만들려면 공식 SDK를 쓴다 (5절 마지막 참조).")

---

## 1. M×N 문제 — 이론편 25.3절

37장에서 만든 도구는 **그 장에서만 쓸 수 있다.** 다른 앱에서 쓰려면 다시 만들어야 한다.

앱이 M개, 도구가 N개면 **M×N개의 연결**이 필요하다.

```
앱 3개 × 도구 10개 = 30개의 연결 코드
```

각 연결마다 인증·오류 처리·데이터 형식을 따로 만들어야 한다.

**MCP는 이를 M+N으로 바꾼다.** 도구를 한 번 만들면 어느 앱에서든 쓸 수 있게 하는 것이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("M x N 문제 (이론편 25.3절)")
print("=" * 70)
print()
print(f"{'앱 수':<10}{'도구 수':<10}{'직접 연결 (MxN)':<20}{'MCP (M+N)':<16}{'절감'}")
print("-" * 70)
for m, n in [(3, 10), (5, 20), (10, 50), (20, 100)]:
    direct = m * n
    mcp = m + n
    print(f"{m:<10}{n:<10}{direct:<20}{mcp:<16}{direct/mcp:.0f}배")
print("-" * 70)
print()
print("규모가 커질수록 차이가 벌어진다.")
print()
print("MCP의 비유: **USB-C**")
print("  예전에는 기기마다 다른 충전 단자가 필요했다.")
print("  하나의 표준이 생기니 케이블 하나로 모든 기기를 쓴다.")

fig, ax = plt.subplots(figsize=(8, 4.5))
sizes = np.arange(2, 21)
ax.plot(sizes, sizes * sizes, marker="o", markersize=3, linewidth=2,
        color="#DC2626", label="직접 연결 (M x N)")
ax.plot(sizes, sizes + sizes, marker="s", markersize=3, linewidth=2,
        color="#0D9488", label="MCP (M + N)")
ax.set_xlabel("앱 수 = 도구 수")
ax.set_ylabel("필요한 연결 수")
ax.set_title("통합 비용 비교")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 2. MCP의 구조 — 이론편 25.3절

MCP는 **Anthropic이 공개한 개방형 표준**이다. 세 역할로 이루어진다.

| 역할 | 하는 일 | 예 |
|---|---|---|
| **Host** | 사용자가 쓰는 앱 | 채팅 앱, IDE |
| **Client** | 서버 하나와의 연결 담당 | Host 안에 서버마다 하나씩 |
| **Server** | 도구·데이터를 제공 | 파일 시스템, DB, 검색 |

```
       ┌─────── Host (앱) ───────┐
       │  Client A  Client B     │
       └──────┬────────┬─────────┘
              │        │
        Server A    Server B
        (파일)      (데이터베이스)
```

### 통신 방식

**JSON-RPC 2.0**을 쓴다. 오래된 표준이라 대부분의 언어에서 다루기 쉽다.

| 전송 방식 | 용도 |
|---|---|
| stdio | 같은 컴퓨터에서 실행되는 서버 |
| HTTP | 원격 서버 |

**같은 JSON 메시지가 두 방식 모두에서 오간다.** 전송 계층만 다르다.

---

## 3. JSON-RPC 메시지 직접 만들기 ★ — 이론편 25.3절

MCP가 특별한 형식을 쓰는 것이 아니다. **JSON-RPC 2.0** 그대로다.

**요청**

```json
{"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
```

**응답**

```json
{"jsonrpc": "2.0", "id": 1, "result": {...}}
```

`id`로 요청과 응답을 짝지으며, 응답이 필요 없는 **알림(notification)**은 `id`를 뺀다.

In [ ]:
import json


def make_request(method, params=None, req_id=1):
    """JSON-RPC 2.0 요청 만들기"""
    msg = {"jsonrpc": "2.0", "id": req_id, "method": method}
    if params is not None:
        msg["params"] = params
    return msg


def make_response(result, req_id=1):
    """정상 응답"""
    return {"jsonrpc": "2.0", "id": req_id, "result": result}


def make_error(code, message, req_id=1, data=None):
    """오류 응답"""
    err = {"code": code, "message": message}
    if data is not None:
        err["data"] = data
    return {"jsonrpc": "2.0", "id": req_id, "error": err}


def make_notification(method, params=None):
    """알림 — id 가 없어 응답을 기대하지 않는다"""
    msg = {"jsonrpc": "2.0", "method": method}
    if params is not None:
        msg["params"] = params
    return msg


print("=" * 70)
print("JSON-RPC 메시지 형태")
print("=" * 70)

print("\n[요청] 도구 목록 조회")
print(json.dumps(make_request("tools/list"), ensure_ascii=False, indent=2))

print("\n[요청] 도구 실행")
print(json.dumps(make_request("tools/call", {
    "name": "calculator",
    "arguments": {"expression": "12 * 7"},
}, req_id=2), ensure_ascii=False, indent=2))

print("\n[응답] 성공")
print(json.dumps(make_response({
    "content": [{"type": "text", "text": "84"}],
}, req_id=2), ensure_ascii=False, indent=2))

print("\n[응답] 오류")
print(json.dumps(make_error(-32602, "Invalid params",
                            req_id=2), ensure_ascii=False, indent=2))

print("\n[알림] id 가 없다")
print(json.dumps(make_notification("notifications/initialized"),
                 ensure_ascii=False, indent=2))

In [ ]:
print("=" * 78)
print("주요 메서드 (이론편 25.3절)")
print("=" * 78)
print()
print(f"{'메서드':<28}{'방향':<16}{'하는 일'}")
print("-" * 78)
methods = [
    ("initialize",              "클라이언트 → 서버", "연결 시작, 기능 협상"),
    ("notifications/initialized", "클라이언트 → 서버", "준비 완료 알림"),
    ("tools/list",              "클라이언트 → 서버", "사용 가능한 도구 목록"),
    ("tools/call",              "클라이언트 → 서버", "도구 실행"),
    ("resources/list",          "클라이언트 → 서버", "읽을 수 있는 자료 목록"),
    ("resources/read",          "클라이언트 → 서버", "자료 읽기"),
    ("prompts/list",            "클라이언트 → 서버", "프롬프트 템플릿 목록"),
    ("prompts/get",             "클라이언트 → 서버", "템플릿 가져오기"),
]
for a, b, c in methods:
    print(f"{a:<28}{b:<16}{c}")
print("-" * 78)
print()
print("이름 규칙이 일정하다: '<대상>/<동작>'")
print("  list 로 목록을 얻고, call·read·get 으로 실제로 쓴다.")
print()
print("표준 오류 코드 (JSON-RPC 2.0)")
print(f"  {-32700:<10} Parse error       JSON 파싱 실패")
print(f"  {-32600:<10} Invalid Request   형식이 잘못됨")
print(f"  {-32601:<10} Method not found  없는 메서드")
print(f"  {-32602:<10} Invalid params    인자가 잘못됨")
print(f"  {-32603:<10} Internal error    서버 내부 오류")

---

## 4. 세 가지 프리미티브 — 이론편 25.3절

MCP 서버가 제공할 수 있는 것은 **세 가지뿐**이다. 이 단순함이 표준의 힘이다.

| 프리미티브 | 성격 | 누가 결정하는가 |
|---|---|---|
| **Tools** | 실행되는 동작 | **모델**이 판단해 호출 |
| **Resources** | 읽기 전용 데이터 | **앱/사용자**가 포함 여부 결정 |
| **Prompts** | 재사용 템플릿 | **사용자**가 선택 |

**"누가 결정하는가"가 다르다**는 점이 중요하다.

In [ ]:
import json

print("=" * 78)
print("세 프리미티브 비교")
print("=" * 78)
print()
print(f"{'항목':<16}{'Tools':<22}{'Resources':<22}{'Prompts'}")
print("-" * 78)
rows = [
    ("성격",     "동작 실행",          "데이터 읽기",         "템플릿"),
    ("제어 주체", "모델",              "앱 또는 사용자",       "사용자"),
    ("부작용",   "있을 수 있음",       "없음 (읽기만)",       "없음"),
    ("식별자",   "이름",              "URI",               "이름"),
    ("예",       "파일 쓰기, API 호출", "파일 내용, DB 레코드", "코드 리뷰 절차"),
]
for a, b, c, d in rows:
    print(f"{a:<16}{b:<22}{c:<22}{d}")
print("-" * 78)
print()

print("[Tools] 37장에서 만든 것과 같다")
print(json.dumps({
    "name": "search_policy",
    "description": "사내 규정을 키워드로 검색합니다.",
    "inputSchema": {
        "type": "object",
        "properties": {"keyword": {"type": "string"}},
        "required": ["keyword"],
    },
}, ensure_ascii=False, indent=2))

print("\n[Resources] URI 로 식별한다")
print(json.dumps({
    "uri": "file:///docs/policy/vacation.md",
    "name": "연차 규정",
    "mimeType": "text/markdown",
}, ensure_ascii=False, indent=2))

print("\n[Prompts] 인자를 받는 템플릿")
print(json.dumps({
    "name": "review_code",
    "description": "코드 리뷰 절차를 따릅니다.",
    "arguments": [
        {"name": "language", "description": "프로그래밍 언어", "required": True},
    ],
}, ensure_ascii=False, indent=2))

### Tools와 Resources를 나눈 이유

둘 다 "데이터를 가져온다"는 점은 비슷해 보인다. **차이는 부작용과 제어권**이다.

| | Tools | Resources |
|---|---|---|
| 모델이 마음대로 부를 수 있나 | 예 | 아니오 |
| 실행하면 무언가 바뀌나 | 그럴 수 있음 | 아니오 |

**"파일 읽기"를 Tool로 만들면** 모델이 판단해서 아무 파일이나 읽으려 할 수 있다.
**Resource로 두면** 앱이 "이 파일을 문맥에 넣겠다"고 결정한다.

37장 7절에서 다룬 **최소 권한 원칙**이 프로토콜 수준에 반영된 것이다.

---

## 5. 간이 MCP 서버 구현 — 이론편 25.3절

이제 **작동하는 서버를 직접 만든다.** 실제 SDK는 전송·인증 등을 더 다루지만,
**핵심 로직은 이것과 같다.**

In [ ]:
import json
import inspect
from datetime import datetime


class MiniMCPServer:
    # 간이 MCP 서버 (이론편 25.3절의 구조를 따른다)
    #
    # 실제 서버와 다른 점:
    #   - 전송 계층(stdio/HTTP) 없이 함수 호출로 대신함
    #   - 인증·세션 관리 생략
    # 같은 점:
    #   - JSON-RPC 2.0 메시지 형식
    #   - tools/list, tools/call 등 메서드 이름
    #   - 세 프리미티브 구조

    PROTOCOL_VERSION = "2025-06-18"

    def __init__(self, name, version="1.0.0"):
        self.name = name
        self.version = version
        self.tools = {}
        self.resources = {}
        self.prompts = {}
        self.initialized = False
        self.log = []

    # ── 등록 ──
    def add_tool(self, func, description=None):
        sig = inspect.signature(func)
        type_map = {str: "string", int: "integer",
                    float: "number", bool: "boolean"}
        props, required = {}, []
        for pname, p in sig.parameters.items():
            props[pname] = {"type": type_map.get(p.annotation, "string")}
            if p.default is inspect.Parameter.empty:
                required.append(pname)

        self.tools[func.__name__] = {
            "func": func,
            "schema": {
                "name": func.__name__,
                "description": description or (func.__doc__ or "").strip().split("\n")[0],
                "inputSchema": {"type": "object", "properties": props,
                                "required": required},
            },
        }

    def add_resource(self, uri, name, content, mime_type="text/plain"):
        self.resources[uri] = {
            "uri": uri, "name": name,
            "mimeType": mime_type, "content": content,
        }

    def add_prompt(self, name, description, template, arguments=None):
        self.prompts[name] = {
            "name": name, "description": description,
            "template": template, "arguments": arguments or [],
        }

    # ── 요청 처리 ──
    def handle(self, request):
        """JSON-RPC 요청을 처리해 응답을 돌려준다"""
        self.log.append(request)

        method = request.get("method")
        params = request.get("params", {})
        req_id = request.get("id")

        # 알림은 응답하지 않는다
        if req_id is None:
            return None

        try:
            if method == "initialize":
                result = self._initialize(params)
            elif method == "tools/list":
                result = {"tools": [t["schema"] for t in self.tools.values()]}
            elif method == "tools/call":
                result = self._call_tool(params)
            elif method == "resources/list":
                result = {"resources": [
                    {k: v for k, v in r.items() if k != "content"}
                    for r in self.resources.values()]}
            elif method == "resources/read":
                result = self._read_resource(params)
            elif method == "prompts/list":
                result = {"prompts": [
                    {k: v for k, v in p.items() if k != "template"}
                    for p in self.prompts.values()]}
            elif method == "prompts/get":
                result = self._get_prompt(params)
            else:
                return make_error(-32601, f"Method not found: {method}", req_id)

            return make_response(result, req_id)

        except ValueError as e:
            return make_error(-32602, str(e), req_id)
        except Exception as e:
            return make_error(-32603, f"{type(e).__name__}: {e}", req_id)

    def _initialize(self, params):
        self.initialized = True
        return {
            "protocolVersion": self.PROTOCOL_VERSION,
            "capabilities": {
                "tools": {"listChanged": False},
                "resources": {"subscribe": False},
                "prompts": {"listChanged": False},
            },
            "serverInfo": {"name": self.name, "version": self.version},
        }

    def _call_tool(self, params):
        name = params.get("name")
        args = params.get("arguments", {})
        if name not in self.tools:
            raise ValueError(f"Unknown tool: {name}")
        output = self.tools[name]["func"](**args)
        return {"content": [{"type": "text", "text": str(output)}],
                "isError": False}

    def _read_resource(self, params):
        uri = params.get("uri")
        if uri not in self.resources:
            raise ValueError(f"Resource not found: {uri}")
        r = self.resources[uri]
        return {"contents": [{"uri": uri, "mimeType": r["mimeType"],
                              "text": r["content"]}]}

    def _get_prompt(self, params):
        name = params.get("name")
        if name not in self.prompts:
            raise ValueError(f"Prompt not found: {name}")
        p = self.prompts[name]
        args = params.get("arguments", {})
        text = p["template"].format(**args) if args else p["template"]
        return {"description": p["description"],
                "messages": [{"role": "user",
                              "content": {"type": "text", "text": text}}]}


print("MiniMCPServer 정의 완료")

In [ ]:
import ast
import operator
from datetime import datetime

# 37장에서 만든 안전한 계산기 재사용
_SAFE_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
             ast.Mult: operator.mul, ast.Div: operator.truediv,
             ast.Pow: operator.pow, ast.USub: operator.neg}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant):
        if not isinstance(node.value, (int, float)):
            raise ValueError("숫자만 허용")
        return node.value
    if isinstance(node, ast.BinOp):
        op = _SAFE_OPS.get(type(node.op))
        if op is None:
            raise ValueError("허용되지 않은 연산자")
        return op(_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):
        op = _SAFE_OPS.get(type(node.op))
        if op is None:
            raise ValueError("허용되지 않은 연산자")
        return op(_safe_eval(node.operand))
    raise ValueError("허용되지 않은 표현식")


def calculator(expression: str) -> str:
    """수식을 계산합니다. 사칙연산과 거듭제곱을 지원합니다."""
    try:
        return str(_safe_eval(ast.parse(expression, mode="eval")))
    except Exception as e:
        return f"계산 오류: {e}"


def get_current_time(timezone: str = "KST") -> str:
    """현재 날짜와 시각을 알려줍니다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S") + f" ({timezone})"


# 서버 구성
server = MiniMCPServer("company-tools", "1.0.0")

server.add_tool(calculator)
server.add_tool(get_current_time)

server.add_resource(
    "policy://vacation", "연차 규정",
    "입사 1년 미만은 월 1일, 1년 이상은 연 15일이 부여됩니다.",
    "text/plain")
server.add_resource(
    "policy://remote-work", "재택근무 규정",
    "재택근무는 주 2회까지 가능하며 팀장 승인이 필요합니다.",
    "text/plain")

server.add_prompt(
    "summarize_policy",
    "규정을 쉽게 풀어 설명합니다.",
    "다음 사내 규정을 신입사원이 이해하기 쉽게 설명해 주세요.\n\n{policy_text}",
    [{"name": "policy_text", "description": "규정 원문", "required": True}])

print("=" * 70)
print("서버 구성 완료")
print("=" * 70)
print(f"이름     : {server.name}")
print(f"Tools    : {len(server.tools)}개 — {list(server.tools.keys())}")
print(f"Resources: {len(server.resources)}개 — {list(server.resources.keys())}")
print(f"Prompts  : {len(server.prompts)}개 — {list(server.prompts.keys())}")

In [ ]:
import json

print("=" * 78)
print("실제 통신 흐름 — 요청과 응답을 눈으로")
print("=" * 78)


def show_exchange(request, label=""):
    """요청과 응답을 나란히 보여준다"""
    if label:
        print(f"\n[{label}]")
    print("  → 요청")
    for line in json.dumps(request, ensure_ascii=False, indent=2).split("\n"):
        print(f"    {line}")

    response = server.handle(request)
    if response is None:
        print("  ← (알림이므로 응답 없음)")
        return None

    print("  ← 응답")
    text = json.dumps(response, ensure_ascii=False, indent=2)
    lines = text.split("\n")
    for line in lines[:14]:
        print(f"    {line}")
    if len(lines) > 14:
        print(f"    ... ({len(lines)}줄 중 14줄)")
    return response


# 1) 연결 시작
show_exchange(make_request("initialize", {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {"name": "my-app", "version": "0.1.0"},
}, req_id=1), "1. 연결 시작")

# 2) 준비 완료 알림
show_exchange(make_notification("notifications/initialized"), "2. 준비 완료 알림")

In [ ]:
import json

# 3) 도구 목록 조회
show_exchange(make_request("tools/list", req_id=2), "3. 도구 목록")

# 4) 도구 실행
show_exchange(make_request("tools/call", {
    "name": "calculator",
    "arguments": {"expression": "13500 * 4 - 10000"},
}, req_id=3), "4. 도구 실행")

# 5) 없는 도구 호출 (오류 처리)
show_exchange(make_request("tools/call", {
    "name": "nonexistent",
    "arguments": {},
}, req_id=4), "5. 없는 도구 (오류)")

In [ ]:
import json

# 6) 자료 목록과 읽기
show_exchange(make_request("resources/list", req_id=5), "6. 자료 목록")

show_exchange(make_request("resources/read", {
    "uri": "policy://vacation",
}, req_id=6), "7. 자료 읽기")

# 8) 프롬프트 템플릿 사용
show_exchange(make_request("prompts/get", {
    "name": "summarize_policy",
    "arguments": {"policy_text": "연차는 연 15일이 부여됩니다."},
}, req_id=7), "8. 프롬프트 템플릿")

print()
print("=" * 78)
print(f"지금까지 주고받은 메시지: {len(server.log)}건")
print()
print("37장의 도구 호출과 무엇이 다른가")
print("  37번: 파이썬 함수를 직접 호출 — 그 주피터 노트북 안에서만 동작")
print("  MCP : JSON 메시지를 주고받음 — 언어·앱이 달라도 동작")

### 실제 MCP 서버를 만들려면

위 구현은 **구조를 이해하기 위한 것**이다. 실제로는 공식 SDK를 쓴다.

```
pip install mcp
```

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("company-tools")

@mcp.tool()
def calculator(expression: str) -> str:
    # 수식을 계산합니다. (실제로는 docstring 을 씁니다)
    return str(safe_eval(expression))

@mcp.resource("policy://vacation")
def vacation_policy() -> str:
    return "연차는 연 15일이 부여됩니다."

if __name__ == "__main__":
    mcp.run()
```

**데코레이터만 붙이면** 스키마 생성·JSON-RPC 처리·전송이 모두 자동이다.
5절에서 손으로 만든 것들을 SDK가 대신해 주는 것이다.

> MCP 사양은 계속 발전 중이다. 최신 내용은 `modelcontextprotocol.io` 에서 확인한다.

---

## 6. Multi-Agent — 왜 나누는가 — 이론편 25.4절

37장에서 만든 Agent는 **하나가 모든 도구를 다뤘다.** 규모가 커지면 문제가 생긴다.

| 문제 | 설명 |
|---|---|
| 도구가 너무 많음 | 스키마만으로 문맥을 크게 차지 |
| 역할 혼동 | 무엇을 우선할지 판단이 흐려짐 |
| 프롬프트 비대 | 모든 지시를 한 곳에 담아야 함 |
| 디버깅 어려움 | 어느 부분이 잘못됐는지 추적 곤란 |

**해법: 역할을 나눈다.** 각 Agent가 자기 몫만 잘하게 하는 것이다.

In [ ]:
import numpy as np

print("=" * 78)
print("단일 Agent vs Multi-Agent (이론편 25.4절)")
print("=" * 78)
print()

# 도구 수에 따른 스키마 토큰 추정
TOKENS_PER_SCHEMA = 80

print(f"{'도구 수':<12}{'단일 Agent 스키마':<24}{'3개로 나눔':<24}{'절감'}")
print("-" * 78)
for n_tools in [5, 15, 30, 60]:
    single = n_tools * TOKENS_PER_SCHEMA
    split = (n_tools // 3) * TOKENS_PER_SCHEMA
    print(f"{n_tools:<12}{single:<24,}{split:<24,}{(1-split/single)*100:.0f}%")
print("-" * 78)
print()
print("각 Agent 는 자기 도구만 알면 되므로 문맥이 가벼워진다.")
print()
print(f"{'항목':<20}{'단일 Agent':<26}{'Multi-Agent'}")
print("-" * 78)
rows = [
    ("도구 수",     "전부",              "역할별로 나눔"),
    ("프롬프트",    "모든 지시를 한 곳에",  "역할별로 짧고 명확"),
    ("디버깅",      "어려움",             "어느 Agent 인지 좁혀짐"),
    ("비용",        "매 호출 전체 스키마",  "필요한 것만"),
    ("복잡도",      "낮음",               "높음 (조율 필요)"),
    ("지연",        "낮음",               "높음 (여러 번 호출)"),
]
for a, b, c in rows:
    print(f"{a:<20}{b:<26}{c}")
print("-" * 78)
print()
print("[주의] Multi-Agent 가 항상 낫지는 않다.")
print("  조율 비용이 들고 호출 횟수가 늘어난다. 8절에서 다룬다.")

---

## 7. 협업 패턴 세 가지 — 이론편 25.4절

Agent들을 어떻게 엮을지에 따라 세 가지 형태가 있다.

In [ ]:
import json


class Agent:
    # 역할을 가진 Agent (실행은 흉내만 낸다)

    def __init__(self, name, role, tools=None):
        self.name = name
        self.role = role
        self.tools = tools or []
        self.history = []

    def work(self, task, context=None):
        """작업 수행 (API 키가 있으면 실제 호출, 없으면 구조만)"""
        record = {"agent": self.name, "task": task, "context": context}
        self.history.append(record)
        return f"[{self.name}] '{task}' 처리 (도구: {', '.join(self.tools) or '없음'})"

    def __repr__(self):
        return f"Agent({self.name}, tools={len(self.tools)})"


# 역할별 Agent
researcher = Agent("연구원", "자료를 찾고 정리한다",
                   ["search_web", "search_policy", "read_document"])
analyst = Agent("분석가", "수치를 계산하고 해석한다",
                ["calculator", "run_python", "make_chart"])
writer = Agent("작성자", "결과를 문서로 정리한다",
               ["format_markdown", "save_file"])

print("=" * 78)
print("패턴 1: 파이프라인 (순차 처리)")
print("=" * 78)
print()
print("  연구원 → 분석가 → 작성자")
print()
print("각 Agent 의 결과가 다음 Agent 의 입력이 된다.")
print()

task = "작년 대비 교육비 지출 변화를 분석해 보고서로 정리해줘"
print(f"작업: {task}")
print()

context = None
for agent in [researcher, analyst, writer]:
    result = agent.work(task, context)
    print(f"  {result}")
    context = result

print()
print("장점: 흐름이 명확하고 디버깅이 쉽다")
print("단점: 앞 단계가 틀리면 뒤가 전부 어긋난다")

In [ ]:
print("=" * 78)
print("패턴 2: 감독자 (Supervisor)")
print("=" * 78)
print()
print("  감독자가 작업을 나누고 적절한 Agent 에게 배분한다")
print()
print("        ┌── 감독자 ──┐")
print("        │      │     │")
print("     연구원  분석가  작성자")
print()


class Supervisor:
    # 작업을 분배하는 감독자 Agent

    def __init__(self, workers):
        self.workers = {w.name: w for w in workers}

    def route(self, task):
        """작업 성격에 따라 담당자를 고른다

        실제로는 LLM이 판단하지만, 여기서는 규칙으로 대신한다.
        """
        if any(k in task for k in ["찾", "검색", "조회", "규정"]):
            return "연구원"
        if any(k in task for k in ["계산", "분석", "얼마", "비교"]):
            return "분석가"
        if any(k in task for k in ["작성", "정리", "문서", "보고서"]):
            return "작성자"
        return "연구원"

    def run(self, tasks):
        results = []
        for t in tasks:
            who = self.route(t)
            worker = self.workers[who]
            print(f"  '{t[:26]}...' → {who}")
            results.append(worker.work(t))
        return results


supervisor = Supervisor([researcher, analyst, writer])

subtasks = [
    "교육비 규정을 찾아줘",
    "작년 지출과 비교해서 계산해줘",
    "결과를 보고서로 정리해줘",
]

print("작업 배분")
supervisor.run(subtasks)

print()
print("장점: 유연하게 배분, 병렬 처리 가능")
print("단점: 감독자가 잘못 배분하면 전체가 어긋난다")

In [ ]:
print("=" * 78)
print("패턴 3: 토론 (Debate)")
print("=" * 78)
print()
print("  같은 문제를 여러 Agent 가 각자 풀고, 결과를 비교한다")
print()

critic = Agent("검토자", "다른 Agent 의 결과를 검토한다", [])

print("흐름")
print("  1) 작성자가 초안을 만든다")
print("  2) 검토자가 문제점을 지적한다")
print("  3) 작성자가 수정한다")
print("  4) 만족스러울 때까지 반복")
print()

draft = writer.work("보고서 초안 작성")
print(f"  1차: {draft}")

for round_num in range(1, 3):
    review = critic.work(f"{round_num}차 검토")
    print(f"  검토 {round_num}: {review}")
    draft = writer.work(f"{round_num}차 수정")
    print(f"  수정 {round_num}: {draft}")

print()
print("장점: 품질이 올라간다 (35번 Self-Consistency 와 비슷한 발상)")
print("단점: 호출 횟수가 크게 늘어난다")
print()
print("=" * 78)
print("세 패턴 비교")
print("=" * 78)
print(f"{'패턴':<16}{'호출 횟수':<16}{'적합한 경우'}")
print("-" * 78)
print(f"{'파이프라인':<16}{'Agent 수만큼':<16}{'단계가 명확한 작업'}")
print(f"{'감독자':<16}{'가변':<16}{'작업 종류가 다양할 때'}")
print(f"{'토론':<16}{'많음 (반복)':<16}{'품질이 중요할 때'}")
print("-" * 78)

---

## 8. 비용과 한계 — 이론편 25.4절

**Multi-Agent가 항상 낫지는 않다.** 오히려 나빠지는 경우도 많다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("Multi-Agent 의 비용")
print("=" * 78)
print()
print("가정: Agent 하나당 호출 1회, 결과 전달에 토큰 200개 추가")
print()

BASE_TOKENS = 500

print(f"{'구성':<24}{'호출 횟수':<14}{'총 토큰(추정)':<18}{'단일 대비'}")
print("-" * 78)
configs = [
    ("단일 Agent", 1, 0),
    ("파이프라인 (3개)", 3, 2),
    ("감독자 + 3개", 4, 3),
    ("토론 (3라운드)", 7, 6),
]
base_total = None
for name, calls, handoffs in configs:
    total = calls * BASE_TOKENS + handoffs * 200
    if base_total is None:
        base_total = total
    print(f"{name:<24}{calls:<14}{total:<18,}{total/base_total:.1f}배")
print("-" * 78)
print()
print("토론 방식은 단일 대비 7배 넘게 든다.")
print("  품질 향상이 그만한 값어치가 있는지 따져야 한다.")

fig, ax = plt.subplots(figsize=(8, 4.5))
names = [c[0] for c in configs]
totals = [c[1]*BASE_TOKENS + c[2]*200 for c in configs]
colors = ["#0D9488", "#1E40AF", "#EA580C", "#DC2626"]
bars = ax.bar(range(len(names)), totals, color=colors)
for b, v in zip(bars, totals):
    ax.text(b.get_x()+b.get_width()/2, v+80, f"{v:,}", ha="center", fontsize=9)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=8, rotation=15, ha="right")
ax.set_ylabel("총 토큰 (추정)")
ax.set_title("구성에 따른 비용")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 78)
print("Multi-Agent 에서 자주 겪는 문제 (이론편 25.4절)")
print("=" * 78)
print()
print(f"{'문제':<22}{'설명':<32}{'대응'}")
print("-" * 78)
issues = [
    ("오류 전파",     "앞 Agent 의 실수가 뒤로 퍼짐",   "각 단계 검증"),
    ("책임 불분명",   "어디서 틀렸는지 찾기 어려움",    "단계별 기록 남기기"),
    ("무한 위임",     "서로 미루며 끝나지 않음",       "최대 단계 제한"),
    ("문맥 손실",     "전달 과정에서 정보가 빠짐",      "핵심 정보 명시적 전달"),
    ("지연 누적",     "순차 실행으로 응답이 느림",      "가능한 것은 병렬로"),
    ("비용 증가",     "호출 횟수에 비례",             "단일로 될 일은 단일로"),
]
for a, b, c in issues:
    print(f"{a:<22}{b:<32}{c}")
print("-" * 78)
print()
print("[먼저 물어야 할 것]")
print()
print("  이 작업이 정말 Agent 를 나눠야 하는가?")
print()
print("  단일 Agent 로 되는 일이라면 그렇게 하는 편이 낫다.")
print("  나누는 것은 **문제가 실제로 생겼을 때** 고려한다.")
print()
print("  37장에서 만든 것도 도구 3개짜리 단일 Agent 였고, 충분히 동작했다.")

In [ ]:
print("=" * 78)
print("언제 나눌 것인가")
print("=" * 78)
print()
print(f"{'신호':<34}{'권장'}")
print("-" * 78)
signals = [
    ("도구가 10개를 넘어감",              "역할별로 분리 고려"),
    ("프롬프트가 지나치게 길어짐",         "분리 고려"),
    ("서로 다른 전문성이 필요",            "분리 (예: 코드 / 법률)"),
    ("어느 단계가 틀렸는지 모르겠음",       "분리해서 추적 가능하게"),
    ("도구 5개 이하로 잘 동작",            "그대로 유지"),
    ("응답 속도가 중요",                  "그대로 유지"),
]
for a, b in signals:
    print(f"{a:<34}{b}")
print("-" * 78)
print()
print("MCP 와 Multi-Agent 의 관계")
print()
print("  MCP        : 도구를 표준화해 **재사용**하게 한다")
print("  Multi-Agent: 역할을 나눠 **복잡도**를 다룬다")
print()
print("  둘은 다른 문제를 푼다. 함께 쓰면 자연스럽다 —")
print("  여러 Agent 가 같은 MCP 서버들을 필요에 따라 골라 쓰는 구조다.")

---

## 9. 정리

### MCP 요약

| 항목 | 내용 |
|---|---|
| 목적 | M×N 통합 문제를 M+N으로 |
| 프로토콜 | **JSON-RPC 2.0** |
| 전송 | stdio(로컬) / HTTP(원격) |
| 프리미티브 | **Tools, Resources, Prompts** 세 가지 |
| 주요 메서드 | `tools/list`, `tools/call`, `resources/read`, `prompts/get` |

**Tools는 모델이, Resources는 앱이, Prompts는 사용자가** 결정한다는 구분이 핵심이다.

### Multi-Agent 요약

| 패턴 | 특징 |
|---|---|
| 파이프라인 | 순차 처리, 흐름 명확 |
| 감독자 | 유연한 배분, 병렬 가능 |
| 토론 | 품질 향상, 비용 큼 |

### 기억할 것

| 항목 | 요점 |
|---|---|
| MCP의 가치 | 한 번 만들어 여러 앱에서 |
| JSON-RPC | `id`로 짝짓기, 알림은 `id` 없음 |
| Tools vs Resources | **부작용과 제어권**의 차이 |
| 실제 구현 | 공식 SDK 사용 (`pip install mcp`) |
| Multi-Agent | **필요할 때만** — 비용과 복잡도 증가 |
| 판단 기준 | 단일로 되면 단일로 |

### 37장과 이어지는 지점

| 37번 | 38번 |
|---|---|
| 함수를 직접 호출 | **JSON 메시지로 통신** |
| 그 장에서만 | 어느 앱에서든 |
| 단일 Agent | 역할 분담 |

### 다음 장

**39. 평가 자동화 — 무엇을 어떻게 측정할까** — 이론편 25.5절.
사람이 일일이 보지 않고도 Agent의 성공 여부를 자동으로 판정하는 방법을 다룬다.
